# 02 — Fixed dataset size → minimum detectable effect

Often, researchers cannot choose the test-set size freely because annotation is expensive.

So the practical question becomes:

> Given the dataset size I already have, what is the smallest difference between methods that I can realistically detect?

This is called the **minimum detectable effect** (MDE).


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from power_utils import *


## Student input

Enter your fixed dataset size and assumptions. This notebook estimates the smallest accuracy improvement that would reach the desired power.


In [ ]:
n_available = ask_int("How many test instances do you have?", 100, minimum=20)
baseline_accuracy = ask_float("Expected accuracy of Model A", 0.90, minimum=0.0, maximum=1.0)
agreement_rate = ask_float("Expected agreement rate in correctness outcomes", 0.95, minimum=0.0, maximum=1.0)
alpha = ask_float("Significance threshold alpha", 0.05, minimum=0.0001, maximum=0.5)
target_power = ask_float("Target power", 0.80, minimum=0.01, maximum=0.99)
n_trials = ask_int("Number of simulation trials", 2000, minimum=100)
delta_step = ask_float("Search step for effect size", 0.001, minimum=0.0001, maximum=0.05)

params = dict(
    n_available=n_available,
    baseline_accuracy=baseline_accuracy,
    agreement_rate=agreement_rate,
    alpha=alpha,
    target_power=target_power,
    n_trials=n_trials,
    delta_step=delta_step,
)
params

## Estimate minimum detectable effect

In [ ]:
mde, mde_power = find_mde_mcnemar(
    n_instances=n_available,
    baseline_accuracy=baseline_accuracy,
    agreement_rate=agreement_rate,
    alpha=alpha,
    target_power=target_power,
    n_trials=n_trials,
    delta_step=delta_step,
)

if mde is None:
    print("No detectable effect found within the search range.")
else:
    print(f"Minimum detectable accuracy improvement: {mde:.2%}")
    print(f"Estimated power at this effect size: {mde_power:.3f}")

## Demonstration: same +1% effect, different dataset sizes

This is the slide example. The effect size is identical, but the statistical reliability changes because the number of examples changes.


In [ ]:
effect_to_check = 0.01
example_sizes = [100, 10_000]
rows = []
for n in example_sizes:
    pwr = estimate_mcnemar_power(
        n_instances=n,
        baseline_accuracy=baseline_accuracy,
        expected_delta=effect_to_check,
        agreement_rate=agreement_rate,
        alpha=alpha,
        n_trials=n_trials,
    )
    rows.append({
        "Test examples": n,
        "Model A": baseline_accuracy,
        "Model B": baseline_accuracy + effect_to_check,
        "Difference": effect_to_check,
        "Estimated power": pwr,
    })

pd.DataFrame(rows)

## MDE curve across dataset sizes

This is useful for explaining why a dataset may be large enough for large effects but not for small improvements.


In [ ]:
n_grid = [50, 100, 250, 500, 1000, 2000, 5000, 10000]
rows = []
for n in n_grid:
    mde_n, power_n = find_mde_mcnemar(
        n_instances=n,
        baseline_accuracy=baseline_accuracy,
        agreement_rate=agreement_rate,
        alpha=alpha,
        target_power=target_power,
        n_trials=max(500, n_trials // 2),
        delta_step=max(delta_step, 0.002),
    )
    rows.append({"n_instances": n, "minimum_detectable_effect": mde_n, "power": power_n})

mde_by_n = pd.DataFrame(rows)
mde_by_n

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(mde_by_n["n_instances"], mde_by_n["minimum_detectable_effect"], marker="o")
plt.xscale("log")
plt.xlabel("Number of test instances")
plt.ylabel("Minimum detectable accuracy improvement")
plt.title("Larger test sets can detect smaller effects")
plt.show()

## Discussion questions

1. If your dataset has only 100 examples, should you claim that a +1% difference is meaningful?
2. If annotation funding limits you to 500 examples, what size of effect should your paper claim it can detect?
3. How would your conclusion change if the models mostly fail on the same items versus on different items?
